# OpenMontage — Chatterbox v2 measurement (FINAL real CC0 voices, 270 scenes, T4)
Clones Chatterbox per channel from the APPROVED real CC0 reference (audio_prompt_path), not the shared default voice. 100% generated, no extrapolation. Self-reporting.

In [ ]:
import os, sys, time, json, torch
from pathlib import Path
TEST_FORCE_T4=True; GPU_BRANCH='unknown'; GPU_NAME='unknown'
if torch.cuda.is_available():
    cc=torch.cuda.get_device_properties(0).major; GPU_NAME=torch.cuda.get_device_name(0)
    if TEST_FORCE_T4:
        if cc<7: print(f'[FATAL] T4 required, got CC={cc}'); sys.exit(1)
        GPU_BRANCH='t4_or_better'
    elif cc>=7: GPU_BRANCH='t4_or_better'
    elif cc==6: GPU_BRANCH='p100'
    else: GPU_BRANCH='old_gpu'
else: print('No GPU'); sys.exit(1)
print(f'GPU: {GPU_NAME} (CC={cc}) branch={GPU_BRANCH}')

In [ ]:
import subprocess, sys
def pip_install(no_deps,*pkgs):
    cmd=[sys.executable,'-m','pip','install','-q']
    if no_deps: cmd.append('--no-deps')
    cmd.extend(pkgs); print('  $ pip install '+' '.join(pkgs),flush=True)
    return subprocess.call(cmd)==0
pip_install(True,'chatterbox-tts')
pip_install(True,'resemble-perth>=1.0.0','conformer==0.3.2','spacy-pkuseg','pykakasi==2.3.0','pyloudnorm','omegaconf','s3tokenizer','librosa==0.11.0','gradio==6.8.0')
pip_install(True,'transformers==5.2.0','diffusers==0.29.0')
print('install done')

In [ ]:
import torch, warnings, time, traceback
warnings.filterwarnings('ignore'); device='cuda'
print('Loading Chatterbox (singleton)...',flush=True)
cb=None; cb_sr=24000
try:
    t0=time.time(); from chatterbox.tts import ChatterboxTTS
    cb=ChatterboxTTS.from_pretrained(device=device); cb_sr=int(cb.sr)
    print(f'  loaded in {round(time.time()-t0,2)}s sr={cb_sr}',flush=True)
except Exception as e: print('load FAILED',repr(e)); traceback.print_exc(); sys.exit(1)

In [ ]:
SCENES = {
  "crime-ledger": [
    [
      "scene_01",
      "February first, 2009. Jeju Island. A childcare teacher boards a taxi at three AM."
    ],
    [
      "scene_02",
      "She calls a taxi through directory service. That is the last confirmed trace of her alive."
    ],
    [
      "scene_03",
      "The taxi arrives. She gets in. The meter activates. She never reaches home."
    ],
    [
      "scene_04",
      "Her family reports her missing after six days. A bag appears. Her wallet. Her phone."
    ],
    [
      "scene_05",
      "Seven days later, a farmer finds her in a drainage ditch near Aewol. Top only."
    ],
    [
      "scene_06",
      "The autopsy shows ligature marks on her neck. Strangulation. Upper clothing missing. Bruising from struggle."
    ],
    [
      "scene_07",
      "But the pathologist estimates death within twenty-four hours of discovery. Not the day she vanished."
    ],
    [
      "scene_08",
      "Police canvass five thousand taxi drivers across Jeju. The focus narrows to one man: Park."
    ],
    [
      "scene_09",
      "Park drove a white NF Sonata. CCTV places a matching vehicle near the scene at the critical time."
    ],
    [
      "scene_10",
      "Analysts find microfibers in Park's taxi. Similar to fur on Lee's jacket. A possible connection."
    ],
    [
      "scene_11",
      "But the fibers are mass-produced. The court rules they cannot be uniquely linked to one vehicle."
    ],
    [
      "scene_12",
      "Park deleted phone call records. Suspicious. But deletion alone does not prove murder."
    ],
    [
      "scene_13",
      "A blood-stained jean is found in a motel room. The search lacked a warrant. The evidence is suppressed."
    ],
    [
      "scene_14",
      "Prosecutors rely on circumstantial evidence. No weapon. No DNA. No eyewitness. No direct proof."
    ],
    [
      "scene_15",
      "The first trial begins in 2019. Ten years after the crime. The family waits. The evidence is weighed."
    ],
    [
      "scene_16",
      "July 11th, 2019. The verdict arrives. Not guilty. The fibers are insufficient. The CCTV is inconclusive."
    ],
    [
      "scene_17",
      "Prosecutors appeal immediately. They argue the cumulative weight of evidence proves guilt."
    ],
    [
      "scene_18",
      "The appellate court reviews. Same evidence. Same arguments. Same conclusion. Not guilty again."
    ],
    [
      "scene_19",
      "July 8th, 2020. The appellate court reaffirms acquittal. Circumstantial evidence cannot exclude reasonable doubt."
    ],
    [
      "scene_20",
      "Prosecutors escalate to the Supreme Court. They challenge the lower courts' standard of proof."
    ],
    [
      "scene_21",
      "October 28th, 2021. The Supreme Court confirms acquittal. Mass-produced fibers cannot single out one suspect."
    ],
    [
      "scene_22",
      "The murder statute of limitations expired. The 2015 abolition came too late for Lee's case."
    ],
    [
      "scene_23",
      "Park had relocated to North Gyeongsang. He lived there for nine years before rearrest. The sources say he moved between various regions. Not an assumed name. Just displacement."
    ],
    [
      "scene_24",
      "After acquittal, Park speaks. His life is destroyed. His defense team claimed he never met Lee. Park himself said his life was ruined and the system treated him like a shackle."
    ],
    [
      "scene_25",
      "The family receives no closure. No conviction. No apology. Just a legal system that reached no answer."
    ],
    [
      "scene_26",
      "In 2015, South Korea abolished the murder statute. Cold case families gained hope."
    ],
    [
      "scene_27",
      "Jeju police form a cold case team in 2016. They re-examine the body. The taxi. The timeline."
    ],
    [
      "scene_28",
      "The timeline remains contested. The autopsy says death within twenty-four hours of discovery. Police say February first."
    ],
    [
      "scene_29",
      "To resolve the timeline, scientists run decomposition experiments. Pigs and dogs. Controlled conditions."
    ],
    [
      "scene_30",
      "The results suggest death closer to February first. But animal-to-human extrapolation is not proof."
    ],
    [
      "scene_31",
      "Police arrest Park in May 2018. Nine years after the crime. They believe evidence now supports prosecution."
    ],
    [
      "scene_32",
      "The trial begins March 2019. Park's defense denies everything. No meeting. No crime. No evidence."
    ],
    [
      "scene_33",
      "Prosecutors present target fiber analysis. The defense calls it imprecise. Mass production undermines uniqueness."
    ],
    [
      "scene_34",
      "CCTV shows a white Sonata. Eighteen such taxis operated on Jeju. Probability is not identity."
    ],
    [
      "scene_35",
      "The deleted phone records suggest cover-up. The defense says routine data management. The court notes suspicion."
    ],
    [
      "scene_36",
      "No murder weapon is found. No DNA connects Park. The fibers are not unique. The CCTV is probabilistic."
    ],
    [
      "scene_37",
      "The first verdict: not guilty. Reasonable doubt cannot be excluded. Circumstantial evidence is not proof."
    ],
    [
      "scene_38",
      "Prosecutors appeal. They argue cumulative weight makes the case. The court disagrees. Each piece remains insufficient."
    ],
    [
      "scene_39",
      "July 2020. Second not guilty. The appellate court finds the prosecution's forensic claims below the proof standard."
    ],
    [
      "scene_40",
      "The prosecution goes to the Supreme Court. They claim lower courts misapplied the standard. Cumulative impact ignored."
    ],
    [
      "scene_41",
      "October 2021. The Supreme Court dismisses the appeal. Mass-produced fibers cannot distinguish one taxi from another."
    ],
    [
      "scene_42",
      "Park's acquittal is final. Double jeopardy prevents retrial. The state's chance to prove guilt is exhausted."
    ],
    [
      "scene_43",
      "But legally final is not factually resolved. The questions remain. Who picked Lee up that night?"
    ],
    [
      "scene_44",
      "Another taxi driver reported picking up a woman matching Lee. This lead was deprioritized."
    ],
    [
      "scene_45",
      "The Supreme Court explicitly noted the second taxi possibility. Reasonable doubt. The victim may have boarded a third vehicle."
    ],
    [
      "scene_46",
      "If the second driver is truthful, Lee was alive after leaving Park's taxi. Everything shifts."
    ],
    [
      "scene_47",
      "The second taxi driver's identity is protected. His vehicle was never examined. His route was never verified."
    ],
    [
      "scene_48",
      "Why was this lead deprioritized? The investigation fixated on Park early. Confirmation bias shaped every step."
    ],
    [
      "scene_49",
      "Five thousand drivers canvassed. Breadth without depth is theatre. The second taxi was never located."
    ],
    [
      "scene_50",
      "The decomposition experiment used pigs and dogs. Defense argued species differences. Translation to human is speculative."
    ],
    [
      "scene_51",
      "Target fiber analysis was groundbreaking. But groundbreaking is not infallible. The court demanded higher standards."
    ],
    [
      "scene_52",
      "Eighteen white Sonata taxis operated on Jeju. CCTV could not isolate Park's specific vehicle. Probability is not identity."
    ],
    [
      "scene_53",
      "Park's deleted records were suspicious. Innocent people delete data too. Suspicion is not proof."
    ],
    [
      "scene_54",
      "Park left Jeju after the first investigation. He lived elsewhere for nine years. The case went cold."
    ],
    [
      "scene_55",
      "The cold case team formed in 2016. They re-examined everything. New tests. New evidence. Not new proof."
    ],
    [
      "scene_56",
      "The decomposition experiment shifted the timeline toward February first. But it did not close the case."
    ],
    [
      "scene_57",
      "The Supreme Court's reasoning is narrow. Mass-produced fibers exclude other taxis. Reasonable doubt remains."
    ],
    [
      "scene_58",
      "Double jeopardy blocks retrial. Even if new evidence emerges. Even if the second driver is found."
    ],
    [
      "scene_59",
      "Lee was twenty-six. A childcare teacher. A daughter. She took a taxi home. That is certain."
    ],
    [
      "scene_60",
      "The investigation expended man-years. Forensic science. Court proceedings. The outcome was not what anyone wanted."
    ],
    [
      "scene_61",
      "Park's life after acquittal is private. No media. No advocacy. Just a man destroyed by accusation."
    ],
    [
      "scene_62",
      "The family's grief is ongoing. No resolution changes February first, 2009. A daughter gone."
    ],
    [
      "scene_63",
      "Jeju's Memories of Murder. The nickname references a Korean film about an unsolved killing. Parallels are unavoidable."
    ],
    [
      "scene_64",
      "No weapon. No DNA. No eyewitness. No confession. The prosecution asked for conviction on inference alone."
    ],
    [
      "scene_65",
      "Five thousand drivers canvassed. But canvassing is not finding. The second taxi was never located."
    ],
    [
      "scene_66",
      "The second taxi driver's statement is sworn. His vehicle is unaccounted for. His route is untraced."
    ],
    [
      "scene_67",
      "Park left Jeju after the first investigation. He lived elsewhere for nine years. The sources describe this as drifting between regions. No assumed name is documented."
    ],
    [
      "scene_68",
      "When the cold case team formed in 2016, hope returned. New tests. New evidence. Not new proof."
    ],
    [
      "scene_69",
      "The animal decomposition experiment placed death closer to February first. But science has margins of error."
    ],
    [
      "scene_70",
      "Target fiber analysis was a Korean forensic first. But innovation does not guarantee conviction."
    ],
    [
      "scene_71",
      "Park weeps in the dock after final acquittal. His life is destroyed. Whether guilty or innocent, the damage stands."
    ],
    [
      "scene_72",
      "The family receives no justice. No conviction. No apology. Just a Supreme Court document explaining why evidence fell short."
    ],
    [
      "scene_73",
      "The case is officially closed. The statute expired. No further prosecution is possible. But closed is not resolved."
    ],
    [
      "scene_74",
      "Jeju's Memories of Murder. The nickname references a Korean film about an unsolved killing. Parallels are unavoidable."
    ],
    [
      "scene_75",
      "The second taxi driver was never identified publicly. His statement remains sealed. His route remains untraced."
    ],
    [
      "scene_76",
      "Lee's case exposed flaws in Korean forensic methodology. It tested the limits of circumstantial evidence."
    ],
    [
      "scene_77",
      "The decomposition experiment was a legal first. It established precedent. It also revealed the method's limitations."
    ],
    [
      "scene_78",
      "The microfibers were suggestive. The court said suggestive is not sufficient. Connection requires certainty."
    ],
    [
      "scene_79",
      "CCTV placed a white Sonata near the scene. Eighteen existed on Jeju. Probability is not identity."
    ],
    [
      "scene_80",
      "Park's deleted records were suspicious. Innocent people delete data too. The standard remains: proof beyond doubt."
    ],
    [
      "scene_81",
      "The court's reasoning is precise. Circumstantial evidence can convict. But it must exclude all reasonable alternatives."
    ],
    [
      "scene_82",
      "Park's post-acquittal statement is brief. His life destroyed. His defense claimed he never met Lee. Whether true or false, the damage to Park stands. His own words were about losing everything."
    ],
    [
      "scene_83",
      "The family attended every hearing. Every appeal. Every Supreme Court session. They watched the system work. They felt it fail."
    ],
    [
      "scene_84",
      "The case fractured trust on Jeju. Drivers questioned. Passengers wary. The island felt more dangerous."
    ],
    [
      "scene_85",
      "No weapon. No DNA. No eyewitness. No confession. The prosecution asked for conviction on inference alone."
    ],
    [
      "scene_86",
      "Five thousand drivers canvassed. That is police work. But canvassing is not finding. The second taxi was never located."
    ],
    [
      "scene_87",
      "The voice of the second taxi driver exists in the record. Anonymous. His vehicle unaccounted for. His route untraced."
    ],
    [
      "scene_88",
      "Lee was twenty-six. A childcare teacher. A daughter. She took a taxi home. That is all anyone knows for certain."
    ],
    [
      "scene_89",
      "The system expended resources. Man-years. Forensic science. Court proceedings. The outcome was not what anyone wanted."
    ],
    [
      "scene_90",
      "The evidence board stands mostly empty. The second taxi driver remains anonymous. The true perpetrator has never been identified. Seventeen years later, that question still waits."
    ]
  ],
  "mythology-slavic": [
    [
      "scene_01",
      "Deep in the Russian forest, something watches. Something ancient. Something conscious."
    ],
    [
      "scene_02",
      "The Slavs called it Leshy. Lord of the forest. Shepherd of every creature between"
    ],
    [
      "scene_03",
      "Leshy is not a god. He is the forest made conscious. A spirit that"
    ],
    [
      "scene_04",
      "Scholars trace Leshy to the Proto-Slavic root les, meaning forest. The word IS the"
    ],
    [
      "scene_05",
      "Afanasyev recorded over five hundred folk tales involving Leshy variants. The spirit appears in"
    ],
    [
      "scene_06",
      "Russian, Ukrainian, Belarusian, Polish, Czech, Slovak. Each culture has its own Leshy name. Same"
    ],
    [
      "scene_07",
      "Leshy appears as a tall man with animal features. Claws. Hooves. A tail. Sometimes"
    ],
    [
      "scene_08",
      "He is never alone. Always accompanied by a black dog or a small black"
    ],
    [
      "scene_09",
      "Leshy can change size. Grow tall as the highest pine. Shrink small as a"
    ],
    [
      "scene_10",
      "He leads travelers astray. Makes them walk circles. Hides their hats. Laughs at their"
    ],
    [
      "scene_11",
      "Not cruel. Just following forest rules. Trespass without respect. Leshy returns the favor by"
    ],
    [
      "scene_12",
      "Offerings were made. First Easter egg. Bread with salt. Sometimes a drop of one's"
    ],
    [
      "scene_13",
      "The contract was written in blood. Kept secret. Breaking it meant losing Leshy's protection."
    ],
    [
      "scene_14",
      "Leshy controls the hunt. Determines success or failure. A hunter who respects the forest"
    ],
    [
      "scene_15",
      "He gambles the animals. Cards against another forest spirit. The losers run. The winners"
    ],
    [
      "scene_16",
      "Leshy's domain is every forest. Every grove. Every wild place where trees grow and"
    ],
    [
      "scene_17",
      "He protects the creatures. Bears. Wolves. Owls. The forest is his property. He defends"
    ],
    [
      "scene_18",
      "Hunters made pacts. Leshy guides herds to their weapons. Ensures clean shots. In return:"
    ],
    [
      "scene_19",
      "The pact required secrecy. Revealing it meant losing protection. Leshy would pursue the betrayer."
    ],
    [
      "scene_20",
      "Ending a contract required a cross under the heel. Burying it. Or shooting an"
    ],
    [
      "scene_21",
      "Leshy is afraid of dogs. Tricolored ones. Of cats. Of firearms loaded with copper"
    ],
    [
      "scene_22",
      "He can be made to laugh. That is another escape. A clever peasant might"
    ],
    [
      "scene_23",
      "Leshy visits the human world. Enters pubs. Drinks vodka. Hires workers. Even courts women."
    ],
    [
      "scene_24",
      "He asks to be taught the harmonica. Requests food. Needs a midwife for his"
    ],
    [
      "scene_25",
      "In gratitude, Leshy gives gifts. ancient craft objects. Takes their place in the army. A"
    ],
    [
      "scene_26",
      "When humans fight, Leshy and forest spirits fight too. Their battles mirror ours. The"
    ],
    [
      "scene_27",
      "Leshy's origins are disputed. Cursed human? Child swapped at birth? Offspring of dharmful and"
    ],
    [
      "scene_28",
      "In folk Christianity, Leshy became a demon. Fallen angel cast to earth. But farmers"
    ],
    [
      "scene_29",
      "The Primary Chronicle mentions Perun. The thunder god. Leshy belongs to the older layer."
    ],
    [
      "scene_30",
      "Veles rules water, earth, and the underworld. Leshy is his terrestrial manifestation. The forest"
    ],
    [
      "scene_31",
      "Leshy is not simply harmful. He punishes transgressors. Not innocents. The forest has its"
    ],
    [
      "scene_32",
      "He leads children home when lost. Shows mushroom gatherers the safe path. Benevolence is"
    ],
    [
      "scene_33",
      "The society converting to Christianity called Leshy a demon. Destroying temples. Toppling idols. But"
    ],
    [
      "scene_34",
      "Prince Vladimir dragged Perun's idol through Kiev. Threw it in the Dnieper. Twelve men"
    ],
    [
      "scene_35",
      "But Leshy was harder to kill. He lived in every tree. Every mushroom. Every"
    ],
    [
      "scene_36",
      "Farmers left porridge for the domovoi. The household spirit. A relative of Leshy. Both"
    ],
    [
      "scene_37",
      "Domovoi and Leshy are siblings. One inside. One outside. Both demand respect. Both punish"
    ],
    [
      "scene_38",
      "Kupala Night. The summer solstice. The veil between worlds thins. Leshy walks among humans."
    ],
    [
      "scene_39",
      "On this night, Leshy can be captured. His power is weakest. But who would"
    ],
    [
      "scene_40",
      "Maslenitsa. The butter week. Spring chasing winter. Leshy is honored, then sent away. The"
    ],
    [
      "scene_41",
      "Every seasonal ritual encodes a relationship with Leshy. Not worship. Negotiation. The Slavs bargained"
    ],
    [
      "scene_42",
      "The World Tree connects three realms. Prav. Yav. Nav. Leshy walks the trunk. Between"
    ],
    [
      "scene_43",
      "Prav is the upper world of gods. Yav is our mortal world. Nav is"
    ],
    [
      "scene_44",
      "Leshy is not confined to one realm. He moves between them. That is why"
    ],
    [
      "scene_45",
      "The Golubinaya Kniga places Leshy in the cosmic order. Not as villain. As function."
    ],
    [
      "scene_46",
      "Nineteenth-century ethnographers documented Leshy beliefs across the Russian Empire. Khudyakov. Dahl. Afanasyev."
    ],
    [
      "scene_47",
      "Afanasyev's Narodnye russkie skazki contains over five hundred tales. Many feature Leshy directly. Others"
    ],
    [
      "scene_48",
      "Levkievskaya's Slavic Antiquities defines leshiy across five volumes. Etymology. Regional variants. Ritual function."
    ],
    [
      "scene_49",
      "Ivanits' Russian Folk Belief traces Leshy from pre-Christian times through Soviet suppression into modern"
    ],
    [
      "scene_50",
      "The Primary Chronicle mentions Perun. The thunder god. Leshy belongs to the older layer."
    ],
    [
      "scene_51",
      "Archaeology shows sacred groves. Wooden idols. Offering pits. The material culture of Leshy worship"
    ],
    [
      "scene_52",
      "Soviet ethnographers documented Leshy beliefs surviving in rural communities. The spirit was not dead."
    ],
    [
      "scene_53",
      "Post-Soviet folklore collections show new variants. Leshy adapting to modern conditions. Still the same"
    ],
    [
      "scene_54",
      "The etymology les is uncontroversial. The suffix -hy denotes lordship. Leshy means Lord of"
    ],
    [
      "scene_55",
      "Regional variants proliferate. Leshy. Lesovik. Leshak. Same spirit. Different name. The forest speaks in"
    ],
    [
      "scene_56",
      "Leshy is described as ugly. Hairy. Hoofed. Yet he has wives. Children. Human desires."
    ],
    [
      "scene_57",
      "He is blamed for missing people. For drowned fishermen. For hunters who never return."
    ],
    [
      "scene_58",
      "The vodyanoy drowns people. The rusalka lures them. Leshy merely misleads. A lesser offense."
    ],
    [
      "scene_59",
      "Rusalka Week is the most dangerous time. Spring. Fertility. The drowned women return. Leshy"
    ],
    [
      "scene_60",
      "Kikimora is Leshy's female counterpart. Spinning. Weaving. The household's hidden weaver. Beautiful if orderly."
    ],
    [
      "scene_61",
      "Poludnitsa. The noon spirit. Beautiful woman appearing at midday. Causes heatstroke. Madness. She is"
    ],
    [
      "scene_62",
      "These spirits are not characters. They are the spiritual ecology of the Slavic landscape."
    ],
    [
      "scene_63",
      "The farmer entering the forest must acknowledge Leshy. The fisherman must appease the vodyanoy."
    ],
    [
      "scene_64",
      "This is not superstition. It is social technology. A code for sustainable human-nature relationships."
    ],
    [
      "scene_65",
      "The code ensured survival. No overhunting. No pollution. No disrespect. The forest survived. The"
    ],
    [
      "scene_66",
      "Christianity arrived with Prince Vladimir in 988. The idols fell. The temples burned. But"
    ],
    [
      "scene_67",
      "Leshy did not die. He retreated. Into the deepest woods. Into the oldest stories."
    ],
    [
      "scene_68",
      "Folk tales preserved him. Under Christian disguise. Saints replaced gods. But the narrative structure"
    ],
    [
      "scene_69",
      "Baba Yaga. The witch. Is she Leshy's counterpart? Or his mother? The tales are"
    ],
    [
      "scene_70",
      "Koschei the Deathless. The immortal villain. His soul is hidden outside his body. Leshy's"
    ],
    [
      "scene_71",
      "The Firebird. Golden feathers. The ultimate prize. Some tales say Leshy guards it. Others"
    ],
    [
      "scene_72",
      "Zorya. The dawn sisters. They open the gate for the Sun. In some tales,"
    ],
    [
      "scene_73",
      "The zmey. The Slavic dragon. Three heads. Fire. Is he Leshy's enemy? Or his"
    ],
    [
      "scene_74",
      "In South Slavic tradition, the zmey is more sympathetic. A lover. A husband. The"
    ],
    [
      "scene_75",
      "The vila are not rusalki. They do not drown men. They marry them. The"
    ],
    [
      "scene_76",
      "Leshy's laughter is the forest's laugh. Through trees. Through wind. Through water. He is"
    ],
    [
      "scene_77",
      "He is shape-shifter. Animal. Human. Whirlwind. Formless. To describe him is to misdescribe him."
    ],
    [
      "scene_78",
      "In Russian byliny, the hero fights Leshy. Sometimes wins. Sometimes loses. The forest is"
    ],
    [
      "scene_79",
      "The byliny preserve older mythology. Before saints. Before moralizers. The hero negotiates with Leshy."
    ],
    [
      "scene_80",
      "Ivanits documents how Soviet industrialization tried to erase these beliefs. Collective farms. Urbanization. Television."
    ],
    [
      "scene_81",
      "But the forest endures. And with it, the memory that something walks beneath the"
    ],
    [
      "scene_82",
      "Contemporary Slavs still report Leshy encounters. Not as literal belief. As inherited memory. The"
    ],
    [
      "scene_83",
      "The leshy existed before Christianity. Before the Slavic state. Before writing itself. He is"
    ],
    [
      "scene_84",
      "We cannot prove Leshy exists. We can prove that Slavs believed he exists. For"
    ],
    [
      "scene_85",
      "Belief is evidence. Not of the spirit. Of the human need to personify the"
    ],
    [
      "scene_86",
      "He is the forest's consciousness. Our projection onto trees. The boundary between self and"
    ],
    [
      "scene_87",
      "The Primary Chronicle begins with chaos. The forest is chaos made place. Leshy is"
    ],
    [
      "scene_88",
      "Every culture has its forest lord. The Wild Man. The Green Man. Leshy is"
    ],
    [
      "scene_89",
      "He is not gone. He never left. The forest still breathes. The trees still"
    ],
    [
      "scene_90",
      "The old ways asked: what do you owe the place that sustains you? Leshy"
    ]
  ],
  "speculative-biology": [
    [
      "scene_01",
      "In the Australian outback, a beetle defends itself with chemistry. Explosive chemistry. One milligram"
    ],
    [
      "scene_02",
      "This is not speculative thought. This is real biology. The bombardier beetle fires boiling"
    ],
    [
      "scene_03",
      "The mechanism is elegant. Two chambers. Hydroquinone and hydrogen peroxide. Mixing triggers explosion. No"
    ],
    [
      "scene_04",
      "The beetle rotates its abdomen. The explosion propels liquid forward. Recoil is absorbed. Directional"
    ],
    [
      "scene_05",
      "Predators learn quickly. Birds. Spiders. Reptiles. One encounter teaches avoidance for life."
    ],
    [
      "scene_06",
      "But here is the question. If one beetle can do this, why not others?"
    ],
    [
      "scene_07",
      "Convergent evolution suggests it should happen elsewhere. Same pressure. Same chemistry. Different solution."
    ],
    [
      "scene_08",
      "What if a spider developed the same mechanism? Not web-building. Not venom. Explosive defense."
    ],
    [
      "scene_09",
      "Or a frog. Skin glands instead of abdominal chambers. A chemical cocktail expelled on"
    ],
    [
      "scene_10",
      "The physics are identical. The biology is different. The outcome is the same: survival"
    ],
    [
      "scene_11",
      "This is speculative biology. Not speculative thought. Not myth. A thought experiment grounded in"
    ],
    [
      "scene_12",
      "The bombardier beetle exists. Its chemistry is documented. Its mechanism is understood. Everything beyond"
    ],
    [
      "scene_13",
      "But speculation has rules. It must respect evolutionary constraint. Energy budgets. Materials available. Predator-prey"
    ],
    [
      "scene_14",
      "A spider cannot store hydrogen peroxide indefinitely. Its exoskeleton is not built for pressure"
    ],
    [
      "scene_15",
      "A frog's skin is permeable. Storing reactive chemicals would poison the host. The beetle's"
    ],
    [
      "scene_16",
      "Yet nature surprises us. The palmetto weevil uses similar chemistry. So does the bloody-nose"
    ],
    [
      "scene_17",
      "This is the signal. Convergent evolution is not rare. It is the rule. Given"
    ],
    [
      "scene_18",
      "So we ask: what other convergent mechanisms remain undiscovered? And what do they tell"
    ],
    [
      "scene_19",
      "The bombardier beetle is a proof of concept. Evolution can build explosives. It did."
    ],
    [
      "scene_20",
      "On other worlds, with different atmospheres, the chemistry changes. But the pressure remains. Predators"
    ],
    [
      "scene_21",
      "A silicon-based lifeform might store oxidizing acids in crystal chambers. A methane-atmosphere creature might"
    ],
    [
      "scene_22",
      "Speculative biology is not predicting the future. It is mapping the space of possible"
    ],
    [
      "scene_23",
      "The bombardier beetle occupies one point in that space. There are quadrillions more. Most"
    ],
    [
      "scene_24",
      "On Earth, we have catalogued two million species. Estimates range from five million to"
    ],
    [
      "scene_25",
      "Each species is an experiment. Evolution running parallel simulations. Some fail. Some persist. The"
    ],
    [
      "scene_26",
      "The experiment took eighty million years. That is how long it took to perfect"
    ],
    [
      "scene_27",
      "During those eighty million years, countless intermediate forms existed. Failed prototypes. Near-misses. The beetle"
    ],
    [
      "scene_28",
      "The chemistry was not designed. It was discovered. Random mutations. Testing. Selection. The successful"
    ],
    [
      "scene_29",
      "Hydroquinone is toxic. Most animals avoid it. The bombardier beetle embraced the toxicity. Turned"
    ],
    [
      "scene_30",
      "This is the engine of innovation. Not intelligence. Not design. Random variation filtered by"
    ],
    [
      "scene_31",
      "The beetle's mechanism has been studied for military applications. Non-lethal weapons. Smoke screens. The"
    ],
    [
      "scene_32",
      "But evolution got there first. Eighty million years of optimization. No grant proposals. No"
    ],
    [
      "scene_33",
      "The beetle stores chemicals at near-boiling temperatures. Its enzymes are heat-resistant. Its membranes are"
    ],
    [
      "scene_34",
      "We call it a beetle. It calls itself a solution. The difference is perspective."
    ],
    [
      "scene_35",
      "There are over five hundred bombardier beetle species. Each with slight variations. Same core"
    ],
    [
      "scene_36",
      "This is not a single invention. It is an innovation wave. Multiple lineages discovering"
    ],
    [
      "scene_37",
      "The wave started in the Cretaceous. Dinosaurs were still alive. The beetle was already"
    ],
    [
      "scene_38",
      "Dinosaurs are gone. The beetle remains. That is the metric. Not intelligence. Not size."
    ],
    [
      "scene_39",
      "The beetle does not think about its chemistry. It does not understand the reaction."
    ],
    [
      "scene_40",
      "Understanding belongs to us. The observers. The documentarians. We impose narrative on biology. The"
    ],
    [
      "scene_41",
      "This is the role of speculative biology. Not to invent monsters. To expand our"
    ],
    [
      "scene_42",
      "The bombardier beetle is not the limit. It is the beginning. A proof that"
    ],
    [
      "scene_43",
      "What other proofs exist? In soil samples. In deep sea vents. In Canopy epiphytes"
    ],
    [
      "scene_44",
      "Every species is a hypothesis tested by eighty million years of data. We are"
    ],
    [
      "scene_45",
      "The beetle reads data differently. It does not analyze. It reacts. The reaction is"
    ],
    [
      "scene_46",
      "We catalog species. We name them. We preserve them. But we do not fully"
    ],
    [
      "scene_47",
      "One secret: why did this mechanism evolve only in beetles? The chemistry is available"
    ],
    [
      "scene_48",
      "The answer may be historical accident. A mutation in one lineage. Timing. Environmental pressure."
    ],
    [
      "scene_49",
      "Evolution is not inevitable. It is contingent. The same world run twice might never"
    ],
    [
      "scene_50",
      "This is the deepest lesson of the beetle. Not its mechanism. Its contingency. We"
    ],
    [
      "scene_51",
      "The beetle's explosion is not anger. It is not malice. It is chemistry responding"
    ],
    [
      "scene_52",
      "For every action there is an equal reaction. The beetle just adds a catalyst."
    ],
    [
      "scene_53",
      "This makes the beetle more than a curiosity. It is a living physics demonstration."
    ],
    [
      "scene_54",
      "Students learn about it. Engineers study it. Military researchers reverse-engineer it. The beetle has"
    ],
    [
      "scene_55",
      "Yet the beetle knows none of this. It does not study itself. It does"
    ],
    [
      "scene_56",
      "This is the humility of biology. The most sophisticated mechanisms belong to creatures that"
    ],
    [
      "scene_57",
      "The beetle's mechanism works at 100 degrees Celsius. Most proteins denature at 40. This"
    ],
    [
      "scene_58",
      "The enzymes are specially adapted. The membranes are reinforced with cross-linked proteins. Every component"
    ],
    [
      "scene_59",
      "Purpose-built by what? Not by a builder. By eighty million years of selection. Each"
    ],
    [
      "scene_60",
      "This is the blind watchmaker. Dawkins' metaphor. The beetle is the watch. It was"
    ],
    [
      "scene_61",
      "Every successful mutation is a lucky accident. Accumulated. Refined. The beetle is eighty million"
    ],
    [
      "scene_62",
      "We look for patterns in biology. We find them. But the pattern is not"
    ],
    [
      "scene_63",
      "The bombardier beetle's pattern works. For eighty million years. In Australian outbacks. In tropical"
    ],
    [
      "scene_64",
      "That persistence is the real miracle. Not the chemistry. Not the mechanism. The fact"
    ],
    [
      "scene_65",
      "In a million years, the beetle may be gone. Replaced by something better. Or"
    ],
    [
      "scene_66",
      "For now, it pops. That is its contribution to the universe. A small popping"
    ],
    [
      "scene_67",
      "We give it names. We study it. We write papers. But the beetle does"
    ],
    [
      "scene_68",
      "This is the proper perspective. We are not the center. The beetle is not"
    ],
    [
      "scene_69",
      "The explosive beetle is one data point. In a dataset of eight hundred million"
    ],
    [
      "scene_70",
      "And yet. One data point is enough to change a worldview. One beetle proves"
    ],
    [
      "scene_71",
      "What else exceeds expectation? In soil. In ice. In deep ocean vents. In places"
    ],
    [
      "scene_72",
      "Speculative biology asks these questions. Not as speculative thought. As hypotheses with constraints. Physics."
    ],
    [
      "scene_73",
      "The constraints are what make it scientific. A creature that violates thermodynamics is speculative"
    ],
    [
      "scene_74",
      "The bombardier beetle pushes boundaries. It stores energy at temperatures that would cook most"
    ],
    [
      "scene_75",
      "How? Specialized proteins. Reinforced membranes. Catalytic enzymes. Eighty million years of optimization."
    ],
    [
      "scene_76",
      "Optimization is not conscious. It is automatic. The beetles that survived had the right"
    ],
    [
      "scene_77",
      "We see the survivor. We do not see the failures. The fossil record is"
    ],
    [
      "scene_78",
      "This is the other lesson. For every successful mechanism, there are a thousand failures."
    ],
    [
      "scene_79",
      "Success is rare. Failure is common. The beetle reminds us that persistence is more"
    ],
    [
      "scene_80",
      "The beetle is not perfect. It has blind spots. Its aim is imperfect. It"
    ],
    [
      "scene_81",
      "Good enough is the standard in biology. Not optimal. Not elegant. Good enough to"
    ],
    [
      "scene_82",
      "The bombardier beetle is good enough. It has been for eighty million years. That"
    ],
    [
      "scene_83",
      "We are new here. One species among eight hundred million years of experiments. The"
    ],
    [
      "scene_84",
      "We could learn from it. Patience. Persistence. Chemical pragmatism. The willingness to be explosive"
    ],
    [
      "scene_85",
      "But we do not learn from beetles. We study them. We write papers. We"
    ],
    [
      "scene_86",
      "Perhaps that is the final lesson. Wisdom does not require self-awareness. It requires only"
    ],
    [
      "scene_87",
      "The beetle pops. The predator retreats. The forest continues. Small explosions in a vast"
    ],
    [
      "scene_88",
      "This is the sound of evolution working. Not a thunderclap. Not a revolution. A"
    ],
    [
      "scene_89",
      "We should listen more carefully. The quietest sounds often carry the most important information."
    ],
    [
      "scene_90",
      "The bombardier beetle is not a monster. It is a proof. That life can"
    ]
  ]
}
CHANNEL_REFS = {
  "crime-ledger": {
    "name": "simon_evers",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/simon_evers.flac",
    "attribution": "LibriVox reader 1255 — Celebration of Dialects and Accents, Vol 2, Track 18 (English, Received Pronunciation)"
  },
  "mythology-slavic": {
    "name": "padraig_o'hiceadha-lyrical",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/padraig_o'hiceadha-lyrical.flac",
    "attribution": "LibriVox reader 2588 — Celebration of Dialects and Accents, Vol 1, Track 2 (Irish, lyrical)"
  },
  "speculative-biology": {
    "name": "nicholas_james_bridgewater",
    "url": "https://raw.githubusercontent.com/OwenTyme/voice-zero/main/voices/nicholas_james_bridgewater.flac",
    "attribution": "LibriVox reader 1618 — Celebration of Dialects and Accents, Vol 2, Track 13 (English, Mid-Atlantic, documentary)"
  }
}
LICENSE = "CC0 1.0 Universal (public-domain dedication) via OwenTyme/voice-zero; source reading public domain on LibriVox"
OUT = Path('/kaggle/working/measure_v2'); OUT.mkdir(parents=True, exist_ok=True)
import urllib.request
import torchaudio as ta
def download(url, path):
    req=urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
    data=urllib.request.urlopen(req, timeout=60).read(); open(path,'wb').write(data); return len(data)
def ref_to_wav(flac, wav, sr=24000):
    w,sr0=ta.load(flac)
    if w.shape[0]>1: w=w.mean(0,keepdim=True)
    if sr0!=sr: w=ta.functional.resample(w,sr0,sr)
    ta.save(str(wav), w, sr)
all_rows={}; channel_stats={}; voice_meta={}; errors=[]
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
for channel, rows in SCENES.items():
    ref=CHANNEL_REFS[channel]; cdir=OUT/channel; cdir.mkdir(parents=True, exist_ok=True)
    ref_flac=cdir/(ref['name']+'.flac'); ref_wav=cdir/(ref['name']+'_ref.wav')
    try:
        download(ref['url'], ref_flac); ref_to_wav(ref_flac, ref_wav)
    except Exception as e:
        errors.append(f'{channel} ref download: '+repr(e)); print('REF FAIL',channel,repr(e),flush=True); continue
    print(f'[{channel}] voice={ref["name"]} ref ready',flush=True)
    gen_total=0.0; aud_total=0.0; scene_rows=[]
    for sid, text in rows:
        t0=time.time()
        try:
            wav=cb.generate(text, audio_prompt_path=str(ref_wav))
            if torch.cuda.is_available(): torch.cuda.synchronize()
            gen=time.time()-t0; dur=int(wav.shape[-1])/cb_sr
            p=cdir/f'{sid}.wav'; ta.save(str(p), wav.cpu(), cb_sr)
            gen_total+=gen; aud_total+=dur
            scene_rows.append({'id':sid,'dur_s':round(dur,3),'gen_s':round(gen,3),'bytes':p.stat().st_size})
        except Exception as e:
            errors.append(f'{channel} {sid}: '+repr(e)); scene_rows.append({'id':sid,'error':repr(e)})
    channel_stats[channel]={'scenes':len(rows),'total_gen_s':round(gen_total,2),'total_audio_s':round(aud_total,2),'rtf':round(aud_total/max(gen_total,1e-6),3)}
    voice_meta[channel]={'voice':ref['name'],'attribution':ref['attribution'],'license':LICENSE,'ref_url':ref['url']}
    all_rows[channel]=scene_rows
    print(f'  channel {channel}: gen={gen_total:.1f}s audio={aud_total:.1f}s rtf={channel_stats[channel]["rtf"]}',flush=True)
manifest={'project_id':'tts-measure-chatterbox-v2-final-voices','gpu_name':GPU_NAME,'gpu_branch':GPU_BRANCH,
          'model_load_s': None,'vram_peak_bytes':int(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else 0,
          'generated_at':time.strftime('%Y-%m-%dT%H:%M:%S'),'errors':errors,'voice_meta':voice_meta,
          'channel_stats':channel_stats,'channels':all_rows}
(OUT/'measure_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps({k:{kk:vv for kk,vv in v.items() if kk!='scenes'} for k,v in channel_stats.items()}, indent=2))
print('VOICE META:'); print(json.dumps(voice_meta, indent=2))